# C2 — Fine-tune Wav2Vec2-large-XLSR-53 phoneme CTC (Kaggle)

Dựng **acoustic model** Wav2Vec2-CTC 39 phone CMU cho Group C. **Không phải thí nghiệm GOP.** Speechocean762 chỉ dùng lúc extract/eval GOP sau này — **không train trên SO762**.

Port của `scripts/finetune_phoneme_ctc.py --backbone wav2vec2`. Recipe AM khớp card C2 / Cao et al. IS24: XLSR-53 + Librispeech train-clean-100 + CTC 39 phone. Group C **không** dùng GOP segmentation-free của Cao; chỉ tái dùng loại AM này.

| | Giá trị |
| --- | --- |
| Pretrained | `facebook/wav2vec2-large-xlsr-53` (encoder đa ngữ; **không** dùng checkpoint ASR chữ cái kiểu jonatasgrosman) |
| Vocab | `<pad>` + 39 CMU phones (cùng `processor_config_gop/vocab.json`) |
| Data | Librispeech **train-clean-100** + **dev-clean** |
| Loss | CTC, freeze feature encoder |
| Default | **5 epoch** (vừa 1 session Kaggle), lr `1e-4`, warmup 500, seed 0, fp16 |

Sau khi train, download zip → giải nén vào `GOP-Empirical-Study/data/wav2vec2_phoneme_ctc` → `extract_ssl_gop.py --model wav2vec2`.

`checkpoint-8000` trong repo Cao thiếu `pytorch_model.bin` (Git LFS). Notebook này train lại AM đó.

XLSR-53 **large** (24 layer, hidden 1024) nặng hơn HuBERT-base ở notebook C3. T4 16GB dễ OOM; batch mặc định nhỏ hơn C3.

**Timeout Kaggle:** run thật `2662/8360` steps (~3 epoch) đã hết ~12 giờ rồi bị kill **khi đang ghi checkpoint** — cell zip không chạy. Notebook này không còn nhắm 10 epoch/1 session. Thay vào đó: 5 epoch (eval_loss đã ~0.19 ở epoch 3), eval trên subset, `group_by_length`, và **callback dừng lúc 10.75h** để kịp `save` + zip. Muốn đủ 10 epoch kiểu Cao: `EPOCHS = 10`, `RESUME = True`, chạy session 2.

---

## Setup Kaggle

1. **Accelerator → GPU** (T4 hoặc P100). CPU sẽ cực chậm.
2. **Internet → On** (download pretrained `wav2vec2-large-xlsr-53`; không cần để lấy wav LS).
3. **Add Input** dataset `kittrngtun/librispeech-asr-corpus` (`train.csv`, `dev.csv`, `train.ctm`, `dev.ctm`, `train-clean-100/`, `dev-clean/`). Cell Config tự gắn path dưới `/kaggle/input`. **Không** download LibriSpeech từ Hugging Face nếu Input đã có.
4. Chạy cell **Config**: `SMOKE = True` lần đầu (vài phút). Khi ổn, `SMOKE = False` rồi **Restart & Run All**.
5. Session GPU Kaggle ~9–12 giờ. Full train XLSR-53 trên T4 thường **10–20 giờ** → resume qua `checkpoint-*`.
6. Output: `/kaggle/working/wav2vec2_phoneme_ctc/` và `/kaggle/working/wav2vec2_phoneme_ctc.zip`.
7. **Disk:** HF cache + `/tmp` mặc định làm đầy system disk → Kaggle kill session (`high /tmp usage` / Docker daemon). Cell setup chuyển cache sang `/kaggle/tmp` (không persist). Checkpoint chỉ ghi `/kaggle/working`.

### Nhãn phoneme (CTC chỉ cần **chuỗi** phone, không cần time-align)

- **`g2p` (mặc định, dễ nhất trên Kaggle):** transcript Librispeech → `g2p_en` (CMU) → bỏ stress. Gần inventory C2 nhưng **không** giống từng token Kaldi CTM.
- **`ctm` (khớp recipe IS24 / script local):** upload CTM 5 cột. Silence bị drop, stress bị strip — giống `finetune_phoneme_ctc.py`.

Không train trên wav Speechocean762.


## 1. Cài package


In [ ]:
!pip install -q "transformers>=4.30" datasets accelerate soundfile librosa g2p-en jiwer

In [ ]:
import os
import shutil
from pathlib import Path


def _scratch_root() -> Path:
    """Prefer existing Kaggle scratch mounts; never create /kaggle/tmp on the root overlay."""
    for cand in (Path("/kaggle/tmp"), Path("/kaggle/temp")):
        if not cand.is_dir():
            continue
        try:
            probe = cand / ".write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return cand
        except OSError:
            continue
    if Path("/kaggle/working").is_dir():
        p = Path("/kaggle/working/_scratch")
        p.mkdir(parents=True, exist_ok=True)
        return p
    p = Path("./_scratch")
    p.mkdir(parents=True, exist_ok=True)
    return p


SCRATCH = _scratch_root()
HF_HOME = SCRATCH / "hf"
TMP_DIR = SCRATCH / "tmp"
NLTK_DIR = SCRATCH / "nltk"
for p in (HF_HOME, TMP_DIR, NLTK_DIR, SCRATCH / "xdg"):
    p.mkdir(parents=True, exist_ok=True)

# Must be set before importing transformers / datasets (they read env at import).
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME / "transformers")
os.environ["XDG_CACHE_HOME"] = str(SCRATCH / "xdg")
os.environ["TMPDIR"] = str(TMP_DIR)
os.environ["TMP"] = str(TMP_DIR)
os.environ["TEMP"] = str(TMP_DIR)
os.environ["NLTK_DATA"] = str(NLTK_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"


def print_disk(tag: str = "") -> None:
    print("disk", tag, "scratch", SCRATCH)
    for p in ("/", "/tmp", "/kaggle/working", "/kaggle/tmp", "/kaggle/temp", str(SCRATCH)):
        path = Path(p)
        if not path.exists():
            continue
        u = shutil.disk_usage(path)
        print(f"  {p}: free={u.free / 1e9:.1f}G used={u.used / 1e9:.1f}G")


print_disk("after-cache-redirect")

import nltk

nltk.data.path.insert(0, str(NLTK_DIR))
for pkg in (
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "cmudict",
    "punkt",
    "punkt_tab",
):
    nltk.download(pkg, download_dir=str(NLTK_DIR), quiet=True)
print("deps ok")

## 2. GPU / phiên bản


In [ ]:
import torch
from transformers import __version__ as transformers_version

print("transformers", transformers_version)
print("cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Bật Accelerator → GPU rồi restart runtime.")
print(torch.cuda.get_device_name(0), "vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 3. Config

Đổi `SMOKE`, `LABEL_SOURCE`, `EPOCHS`, `BATCH_SIZE` tại đây. Cell train đọc các biến này.

XLSR-53 large: T4 dùng `BATCH_SIZE=2` (OOM thì `1`). P100/V100 có thể `4`.


In [ ]:
from pathlib import Path
import time

# True = ~128 utt, 1 epoch, kiểm tra pipeline. False = full train-clean-100.
SMOKE = True
LABEL_SOURCE = "g2p"  # overwritten to "ctm" if LS CTM files are found

PRETRAINED = "facebook/wav2vec2-large-xlsr-53"
SAMPLING_RATE = 16000
# 10 epoch trên T4 ≈ 37h (log: 2662/8360 steps trong ~12h rồi timeout).
# 5 epoch vừa 1 session sau khi bỏ full-dev eval + group_by_length.
# Eval_loss đã ~0.19 ở epoch 3; 5 epoch đủ AM cho GOP. Muốn 10: EPOCHS=10 + RESUME.
EPOCHS = 1.0 if SMOKE else 5.0
# T4 16GB + XLSR-53 large: 2×8 = effective 16 (× n_gpu). OOM → BATCH_SIZE=1, GRAD_ACCUM=16.
# Cao IS24 local: 16×2 (máy lớn hơn). Không cố batch đó trên T4.
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 1e-4
WARMUP_STEPS = 50 if SMOKE else 500
SAVE_STEPS = 100 if SMOKE else 1000
EVAL_STEPS = SAVE_STEPS
LOGGING_STEPS = 20 if SMOKE else 100
SEED = 0
MAX_SECONDS = 16.0
MAX_TRAIN = 128 if SMOKE else None
MAX_DEV = 32 if SMOKE else None
# Full dev-clean (~2.7k) mỗi 500 bước ≈ 20–30 phút/lần → timeout. Eval subset khi train.
EVAL_MAX = 64 if SMOKE else 256
GRADIENT_CHECKPOINTING = True
GROUP_BY_LENGTH = True  # OOM (hai câu dài chung batch) → False
DATALOADER_WORKERS = 0  # Kaggle Jupyter + Dataset custom: 2 dễ deadlock
# Dừng trước kill ~12h để cell zip chạy. Đo từ SESSION_T0 (cell setup).
SESSION_MAX_HOURS = 10.75

ON_KAGGLE = Path("/kaggle/working").is_dir()
OUTPUT_DIR = Path("/kaggle/working/wav2vec2_phoneme_ctc" if ON_KAGGLE else "outputs/wav2vec2_phoneme_ctc")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path(".")


def find_ls_root() -> Path | None:
    """Kaggle Input / local corpus with train.csv+dev.csv. Never require Hugging Face for wavs."""

    def is_root(p: Path) -> bool:
        return p.is_dir() and (p / "train.csv").is_file() and (p / "dev.csv").is_file()

    named = [
        Path("/kaggle/input/datasets/kittrngtun/librispeech-asr-corpus"),
        Path("/kaggle/input/kittrngtun/librispeech-asr-corpus"),
        Path("/kaggle/input/librispeech-asr-corpus"),
        Path("../data/LibriSpeech ASR corpus"),
        Path("data/LibriSpeech ASR corpus"),
        Path("LibriSpeech ASR corpus"),
    ]
    hits: list[Path] = []
    for p in named:
        if is_root(p):
            hits.append(p)
        elif p.is_dir():
            hits.extend(child for child in p.iterdir() if is_root(child))
    if not hits and INPUT_ROOT.is_dir():
        for pattern in ("*/train.csv", "*/*/train.csv", "*/*/*/train.csv", "*/*/*/*/train.csv"):
            for csv in INPUT_ROOT.glob(pattern):
                parent = csv.parent
                if (parent / "dev.csv").is_file():
                    hits.append(parent)
            if hits:
                break
    if not hits:
        return None
    with_ctm = [p for p in hits if (p / "train.ctm").is_file() and (p / "dev.ctm").is_file()]
    return (with_ctm or hits)[0]


LS_ROOT = find_ls_root()
TRAIN_CSV = (LS_ROOT / "train.csv") if LS_ROOT is not None else None
DEV_CSV = (LS_ROOT / "dev.csv") if LS_ROOT is not None else None
TRAIN_CTM = (LS_ROOT / "train.ctm") if LS_ROOT is not None and (LS_ROOT / "train.ctm").is_file() else None
DEV_CTM = (LS_ROOT / "dev.ctm") if LS_ROOT is not None and (LS_ROOT / "dev.ctm").is_file() else None
if TRAIN_CTM is not None and DEV_CTM is not None:
    LABEL_SOURCE = "ctm"

RESUME = True  # tiếp tục checkpoint-* mới nhất trong OUTPUT_DIR nếu có
print("kaggle", ON_KAGGLE, "out", OUTPUT_DIR, "label", LABEL_SOURCE, "smoke", SMOKE, "pretrained", PRETRAINED, "ls_root", LS_ROOT, "train_csv", TRAIN_CSV, "train_ctm", TRAIN_CTM)
if "SCRATCH" in globals():
    print("scratch", SCRATCH)

## 4. Vocab 39 phone (frozen, giống C3 / `processor_config_gop`)

Không thêm token. `vocab_size = 40` (`<pad>` + 39).


In [ ]:
import json
import re
from collections import OrderedDict
from dataclasses import dataclass
from typing import Any

SCORED_PHONES = [
    "AA", "AE", "AH", "AO", "AW", "AY", "B", "CH", "D", "DH",
    "EH", "ER", "EY", "F", "G", "HH", "IH", "IY", "JH", "K",
    "L", "M", "N", "NG", "OW", "OY", "P", "R", "S", "SH",
    "T", "TH", "UH", "UW", "V", "W", "Y", "Z", "ZH",
]
FROZEN_VOCAB = {"<pad>": 0, **{p: i for i, p in enumerate(SCORED_PHONES, start=1)}}
assert len(FROZEN_VOCAB) == 40 and FROZEN_VOCAB["ZH"] == 39

_POS_MARKER_RE = re.compile(r"_(B|E|I|S)$")
_STRESS_RE = re.compile(r"[_\d].*")
_SKIP = {"SIL", "SPN", "NSN", "<eps>", "<UNK>", "sil", "spn", "nsn"}
# g2p_en / CMUdict cũ đôi khi ra symbol ngoài 39-phone set.
_G2P_MAP = {"AX": "AH", "IX": "IH", "DX": "D", "EL": "L", "EM": "M", "EN": "N", "UX": "UW", "Q": None}

(OUTPUT_DIR / "vocab.json").write_text(json.dumps(FROZEN_VOCAB, indent=2) + "\n", encoding="utf-8")
print("wrote", OUTPUT_DIR / "vocab.json", "n=", len(FROZEN_VOCAB))


def clean_phone(symbol: str) -> str | None:
    # Same rules as gop_empirical.acoustic.alignment.clean_phone (no local package).
    base = _POS_MARKER_RE.sub("", str(symbol))
    stripped = _STRESS_RE.sub("", base)
    if stripped in _SKIP or base in _SKIP:
        return None
    mapped = _G2P_MAP.get(stripped, stripped)
    if mapped is None:
        return None
    if mapped not in FROZEN_VOCAB:
        return None
    return mapped


def read_phoneme_ctm(path: Path) -> dict[str, list[str]]:
    trans: dict[str, list[str]] = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            if len(parts) < 5:
                raise ValueError(f"{path}: expected Kaldi CTM ≥5 fields, got {parts!r}")
            utt, phone = parts[0], parts[4]
            cleaned = clean_phone(phone)
            if cleaned is None:
                continue
            trans.setdefault(utt, []).append(cleaned)
    return trans


def find_ctm(kind: str) -> Path | None:
    explicit = TRAIN_CTM if kind == "train" else DEV_CTM
    if explicit is not None:
        p = Path(explicit)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    names = {
        "train": ("train.ctm", "ls100_train.ctm", "train-clean-100.ctm"),
        "dev": ("dev.ctm", "ls100_dev.ctm", "dev-clean.ctm", "valid.ctm"),
    }[kind]
    hits = [p for p in INPUT_ROOT.rglob("*.ctm") if p.name.lower() in names]
    return hits[0] if hits else None

## 5. Load Librispeech + gắn chuỗi phone

Ưu tiên wav **local / Kaggle Input** (`train.csv` + `dev.csv` + CTM). Không `.map()` audio thành Arrow float32. `SMOKE` dừng sau `MAX_TRAIN` / `MAX_DEV` câu. Hugging Face `load_dataset` chỉ khi không tìm thấy corpus.


In [ ]:
from datasets import Audio, load_dataset
import pandas as pd
import soundfile as sf

_RE_UTT = re.compile(r"(.*/)*(.*)\.(.*$)")
MAX_SAMPLES = int(MAX_SECONDS * SAMPLING_RATE)
_HF_CACHE = str(HF_HOME / "datasets") if "HF_HOME" in globals() else None


def load_librispeech(split: str):
    errors = []
    kwargs = {"cache_dir": _HF_CACHE} if _HF_CACHE else {}
    for repo in ("librispeech_asr", "openslr/librispeech_asr"):
        try:
            ds = load_dataset(repo, "clean", split=split, **kwargs)
            try:
                return ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE, decode=False))
            except TypeError:
                return ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
        except Exception as exc:
            errors.append(f"{repo}: {exc}")
    raise RuntimeError("Không load được Librispeech:\n" + "\n".join(errors))


def phones_from_g2p(text: str, g2p) -> list[str] | None:
    phones = [p for tok in g2p(text) if (p := clean_phone(tok)) is not None]
    return phones or None


def phones_from_ctm(utt: str, trans: dict[str, list[str]]) -> list[str] | None:
    return trans.get(utt) or trans.get("lbi-" + utt)


def utt_from_path(path: str) -> str:
    m = _RE_UTT.match(path)
    return m.group(2) if m else Path(path).stem


def audio_path_of(ex: dict) -> str | None:
    for key in ("file", "path"):
        val = ex.get(key)
        if isinstance(val, str) and val:
            return val
    audio = ex.get("audio")
    if isinstance(audio, dict) and audio.get("path"):
        return str(audio["path"])
    if isinstance(audio, str) and audio:
        return audio
    return None


def n_samples_of(path: str, ex: dict) -> int | None:
    try:
        return int(sf.info(path).frames)
    except Exception:
        audio = ex.get("audio")
        if isinstance(audio, dict) and audio.get("array") is not None:
            return len(audio["array"])
        return None


def index_from_hf(raw_ds, trans: dict[str, list[str]] | None, max_keep: int | None):
    """Path + phones only. Never write decoded wav float32 into Arrow cache."""
    rows: list[dict] = []
    skipped = 0
    for ex in raw_ds:
        path = audio_path_of(ex)
        if not path:
            skipped += 1
            continue
        n = n_samples_of(path, ex)
        if n is None or n > MAX_SAMPLES or n <= 0:
            skipped += 1
            continue
        utt = str(ex.get("id") or utt_from_path(path))
        if trans is not None:
            phones = phones_from_ctm(utt, trans)
        else:
            phones = phones_from_g2p(ex["text"], g2p)
        if not phones:
            skipped += 1
            continue
        rows.append({"path": path, "phones": phones, "utt_id": utt, "n_samples": n})
        if max_keep is not None and len(rows) >= max_keep:
            break
    return rows, skipped


def index_from_csv(csv_path: Path, trans: dict[str, list[str]], max_keep: int | None):
    csv_path = csv_path.resolve()
    csv_dir = csv_path.parent
    df = pd.read_csv(csv_path)
    col = "file_name" if "file_name" in df.columns else "path"
    rows: list[dict] = []
    skipped = 0
    for p in df[col].tolist():
        path = Path(str(p))
        if not path.is_absolute():
            path = (csv_dir / path).resolve()
        path_s = str(path)
        n = n_samples_of(path_s, {})
        if n is None or n > MAX_SAMPLES or n <= 0:
            skipped += 1
            continue
        utt = utt_from_path(path_s)
        phones = phones_from_ctm(utt, trans)
        if not phones:
            skipped += 1
            continue
        rows.append({"path": path_s, "phones": phones, "utt_id": utt, "n_samples": n})
        if max_keep is not None and len(rows) >= max_keep:
            break
    return rows, skipped


g2p = None
train_trans = None
dev_trans = None

if LABEL_SOURCE == "g2p":
    from g2p_en import G2p

    g2p = G2p()
    print("g2p_en ready")
elif LABEL_SOURCE == "ctm":
    train_ctm = find_ctm("train")
    dev_ctm = find_ctm("dev")
    if train_ctm is None or dev_ctm is None:
        raise FileNotFoundError(
            "LABEL_SOURCE='ctm' cần train.ctm và dev.ctm (Add Input) hoặc set TRAIN_CTM/DEV_CTM."
        )
    print("CTM train", train_ctm)
    print("CTM dev", dev_ctm)
    train_trans = read_phoneme_ctm(train_ctm)
    dev_trans = read_phoneme_ctm(dev_ctm)
    print("utt train/dev", len(train_trans), len(dev_trans))
else:
    raise ValueError(LABEL_SOURCE)

if TRAIN_CSV is not None:
    if DEV_CSV is None:
        raise ValueError("TRAIN_CSV đã set thì cũng cần DEV_CSV")
    if train_trans is None or dev_trans is None:
        raise ValueError(
            f"Corpus local {TRAIN_CSV.parent} cần train.ctm và dev.ctm — không fallback Hugging Face."
        )
    print("audio from local CSV (no Hugging Face download)", TRAIN_CSV, DEV_CSV)
    train_rows, skip_tr = index_from_csv(Path(TRAIN_CSV), train_trans, MAX_TRAIN)
    dev_rows, skip_dv = index_from_csv(Path(DEV_CSV), dev_trans, MAX_DEV)
else:
    print("WARNING: không thấy train.csv trên Kaggle Input / data/ — fallback Hugging Face")
    train_raw = load_librispeech("train.100")
    dev_raw = load_librispeech("validation")
    print("loaded librispeech", len(train_raw), "train", len(dev_raw), "dev — indexing paths (no audio map)")
    train_rows, skip_tr = index_from_hf(train_raw, train_trans, MAX_TRAIN)
    dev_rows, skip_dv = index_from_hf(dev_raw, dev_trans, MAX_DEV)
    del train_raw, dev_raw

print("kept train", len(train_rows), "skipped", skip_tr, "dev", len(dev_rows), "skipped", skip_dv)
if not train_rows or not dev_rows:
    raise RuntimeError("Không còn utterance sau filter phone/duration.")
print("example", train_rows[0]["utt_id"], train_rows[0]["phones"][:12], "n_samples", train_rows[0]["n_samples"])
if "print_disk" in globals():
    print_disk("after-index")


## 6. Processor + Wav2Vec2ForCTC (head 40 lớp)

`ignore_mismatched_sizes=True` vì XLSR-53 pretrained không có CTC head 39 phone.


In [ ]:
from transformers import (
    AutoFeatureExtractor,
    AutoModelForCTC,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2Processor,
)

tokenizer = Wav2Vec2CTCTokenizer(
    str(OUTPUT_DIR / "vocab.json"),
    unk_token=None,
    pad_token="<pad>",
    word_delimiter_token=None,
    bos_token=None,
    eos_token=None,
)
_model_kwargs = {}
if "HF_HOME" in globals():
    _model_kwargs["cache_dir"] = str(HF_HOME / "transformers")

feature_extractor = AutoFeatureExtractor.from_pretrained(PRETRAINED, **_model_kwargs)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(OUTPUT_DIR)

model = AutoModelForCTC.from_pretrained(
    PRETRAINED,
    ctc_loss_reduction="mean",
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=len(FROZEN_VOCAB),
    ignore_mismatched_sizes=True,
    **_model_kwargs,
)
if hasattr(model, "freeze_feature_encoder"):
    model.freeze_feature_encoder()
elif hasattr(model, "freeze_feature_extractor"):
    model.freeze_feature_extractor()
if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print("vocab", len(tokenizer), "lm_head", model.lm_head.out_features, "pad", tokenizer.pad_token_id, "params_m", round(n_params, 1))
assert len(tokenizer) == 40 and model.lm_head.out_features == 40
assert model.config.model_type == "wav2vec2"
assert int(model.config.hidden_size) == 1024
assert int(model.config.num_hidden_layers) == 24

## 7. Dataset lazy (decode lúc train) + collator


In [ ]:
import numpy as np
import soundfile as sf
import torch


class PhonemeCTCDataset(torch.utils.data.Dataset):
    """Decode FLAC on the fly so Arrow/HF cache never stores float32 wav copies."""

    def __init__(self, rows: list[dict], processor, sampling_rate: int):
        self.rows = rows
        self.processor = processor
        self.sampling_rate = sampling_rate

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict:
        ex = self.rows[idx]
        arr, sr = sf.read(ex["path"], dtype="float32")
        if arr.ndim > 1:
            arr = arr.mean(axis=1)
        if sr != self.sampling_rate:
            n_tgt = int(round(len(arr) * self.sampling_rate / sr))
            t_old = np.linspace(0.0, 1.0, num=len(arr), endpoint=False)
            t_new = np.linspace(0.0, 1.0, num=n_tgt, endpoint=False)
            arr = np.interp(t_new, t_old, arr).astype(np.float32)
        input_values = self.processor(audio=arr, sampling_rate=self.sampling_rate).input_values[0]
        labels = self.processor(text=ex["phones"], is_split_into_words=True).input_ids
        return {"input_values": input_values, "labels": labels}


train_ds = PhonemeCTCDataset(train_rows, processor, SAMPLING_RATE)
dev_ds = PhonemeCTCDataset(dev_rows, processor, SAMPLING_RATE)
ex0 = train_ds[0]
print("train", len(train_ds), "dev", len(dev_ds))
print("input_values", len(ex0["input_values"]), "labels", ex0["labels"][:16])


@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: bool = True

    def __call__(self, features: list[dict]) -> dict:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=True, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch


collator = DataCollatorCTCWithPadding(processor=processor)


## 8. Train

`eval_strategy` / `evaluation_strategy` được chọn theo phiên bản `transformers` trên Kaggle. Resume tự động nếu còn `checkpoint-*`.


In [ ]:
import inspect
import time

import torch
from transformers import Trainer, TrainerCallback, TrainingArguments

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


class _LengthGroupedSampler(torch.utils.data.Sampler):
    """Shuffle then sort within mega-batches so padded CTC batches stay short."""

    def __init__(self, lengths: list[int], batch_size: int):
        self.lengths = lengths
        self.batch_size = max(1, int(batch_size))

    def __len__(self) -> int:
        return len(self.lengths)

    def __iter__(self):
        n = len(self.lengths)
        indices = torch.randperm(n).tolist()
        mega = self.batch_size * 8
        grouped: list[int] = []
        for start in range(0, n, mega):
            chunk = indices[start : start + mega]
            chunk.sort(key=lambda i: self.lengths[i], reverse=True)
            grouped.extend(chunk)
        return iter(grouped)


class _KaggleTimeLimitCallback(TrainerCallback):
    """Stop before Kaggle kills the kernel so save/zip cells still run."""

    def __init__(self, deadline_monotonic: float):
        self.deadline = deadline_monotonic

    def on_step_end(self, args, state, control, **kwargs):
        if time.monotonic() < self.deadline:
            return control
        print(
            f"time limit reached at step {state.global_step}; "
            "stopping to save checkpoint and zip",
            flush=True,
        )
        control.should_training_stop = True
        control.should_save = True
        control.should_evaluate = False
        return control


class _LengthGroupedTrainer(Trainer):
    def _get_train_sampler(self, *args, **kwargs):
        if not GROUP_BY_LENGTH:
            return super()._get_train_sampler(*args, **kwargs)
        lengths = getattr(self.train_dataset, "lengths", None)
        if not lengths:
            return super()._get_train_sampler(*args, **kwargs)
        n_gpu = max(1, int(getattr(self.args, "n_gpu", 1) or 1))
        return _LengthGroupedSampler(lengths, self.args.per_device_train_batch_size * n_gpu)


sig = inspect.signature(TrainingArguments.__init__)
eval_key = "eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"

n_eval = min(int(EVAL_MAX), len(dev_ds))
eval_ds = torch.utils.data.Subset(dev_ds, range(n_eval))

deadline = SESSION_T0 + float(SESSION_MAX_HOURS) * 3600.0
remain_h = (deadline - time.monotonic()) / 3600.0
n_gpu = max(1, torch.cuda.device_count())
eff_batch = BATCH_SIZE * GRAD_ACCUM * n_gpu
print(
    "n_gpu", n_gpu, "eff_batch", eff_batch, "train", len(train_ds), "eval_subset", n_eval,
    "group_by_length", GROUP_BY_LENGTH, "session_remain_h", round(remain_h, 2),
)
if remain_h < 0.25:
    raise SystemExit(f"Còn {remain_h:.2f}h — quá ít để train. Restart session rồi Run All.")

ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    logging_steps=LOGGING_STEPS,
    fp16=True,
    seed=SEED,
    report_to=[],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_num_workers=int(DATALOADER_WORKERS),
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    remove_unused_columns=False,
)
ta_kwargs[eval_key] = "steps"
if "save_strategy" in sig.parameters:
    ta_kwargs["save_strategy"] = "steps"
if "ignore_data_skip" in sig.parameters:
    ta_kwargs["ignore_data_skip"] = True
if "dataloader_pin_memory" in sig.parameters:
    ta_kwargs["dataloader_pin_memory"] = True
if "eval_do_concat_batches" in sig.parameters:
    ta_kwargs["eval_do_concat_batches"] = False

training_args = TrainingArguments(**ta_kwargs)

ckpt_dirs = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=lambda p: p.stat().st_mtime)
resume_from = str(ckpt_dirs[-1]) if (RESUME and ckpt_dirs) else None
print("resume_from", resume_from)
if "print_disk" in globals():
    print_disk("before-train")

trainer_kwargs = dict(
    model=model,
    data_collator=collator,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    callbacks=[_KaggleTimeLimitCallback(deadline)],
)
trainer_sig = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = processor
else:
    trainer_kwargs["tokenizer"] = processor

trainer = _LengthGroupedTrainer(**trainer_kwargs)
train_out = trainer.train(resume_from_checkpoint=resume_from)
print(train_out.metrics)
print(
    "global_step", trainer.state.global_step,
    "max_steps", trainer.state.max_steps,
    "elapsed_h", round((time.monotonic() - SESSION_T0) / 3600.0, 2),
)

## 9. Lưu checkpoint + zip để download


In [ ]:
import shutil

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
(OUTPUT_DIR / "vocab.json").write_text(json.dumps(FROZEN_VOCAB, indent=2) + "\n", encoding="utf-8")

meta = {
    "backbone": "wav2vec2",
    "pretrained": PRETRAINED,
    "label_source": LABEL_SOURCE,
    "smoke": SMOKE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "lr": LR,
    "warmup_steps": WARMUP_STEPS,
    "seed": SEED,
    "vocab_size": 40,
    "n_train": len(train_ds),
    "n_dev": len(dev_ds),
    "note": "Phoneme CTC AM for Group C2. Not GOP. Copy this folder to data/wav2vec2_phoneme_ctc.",
}
(OUTPUT_DIR / "finetune_meta.json").write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

# Zip chỉ file model/processor, bỏ checkpoint-* cho nhẹ.
zip_root = Path("/kaggle/working" if ON_KAGGLE else OUTPUT_DIR.parent) / "wav2vec2_phoneme_ctc_export"
if zip_root.exists():
    shutil.rmtree(zip_root)
zip_root.mkdir(parents=True)
keep_names = {
    "config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "vocab.json",
    "finetune_meta.json",
    "training_args.bin",
    "model.safetensors",
    "pytorch_model.bin",
    "added_tokens.json",
}
for p in OUTPUT_DIR.iterdir():
    if p.is_file() and (p.name in keep_names or p.suffix in {".json", ".txt"}):
        shutil.copy2(p, zip_root / p.name)
    elif p.is_file() and p.name.startswith("model") and p.suffix in {".bin", ".safetensors"}:
        shutil.copy2(p, zip_root / p.name)

archive = Path("/kaggle/working" if ON_KAGGLE else OUTPUT_DIR.parent) / "wav2vec2_phoneme_ctc"
shutil.make_archive(str(archive), "zip", zip_root)

# HF cache under /kaggle/working would be included in Save Version and can exceed 20GB.
if "SCRATCH" in globals() and str(SCRATCH).startswith("/kaggle/working"):
    print("clearing scratch under /kaggle/working", SCRATCH)
    shutil.rmtree(SCRATCH, ignore_errors=True)

if "print_disk" in globals():
    print_disk("after-zip")
print("saved", OUTPUT_DIR)
print("zip", archive.with_suffix(".zip"), "bytes", archive.with_suffix(".zip").stat().st_size)
print("files", sorted(p.name for p in zip_root.iterdir()))


## 10. Dùng local (sau Kaggle)

1. Download `wav2vec2_phoneme_ctc.zip` từ Output của session.
2. Giải nén vào `GOP-Empirical-Study/data/wav2vec2_phoneme_ctc/` (phải có `config.json`, `vocab.json`, `preprocessor_config.json`, và `model.safetensors` hoặc `pytorch_model.bin`).
3. `configs/c_acoustic_model.yaml` đã trỏ:

```yaml
wav2vec2_checkpoint: data/wav2vec2_phoneme_ctc
wav2vec2_processor: data/wav2vec2_phoneme_ctc
```

4. Extract GOP trên **Speechocean762** (không phải LS-100):

```text
python scripts/extract_ssl_gop.py --config configs/c_acoustic_model.yaml --model wav2vec2
python scripts/run_experiment.py --config configs/c_acoustic_model.yaml --models C1 C2
```

Wav2Vec2 embeddings **không** phải GOP. C2 dùng posterior CTC 39 phone, cùng Kaldi CTM và công thức GOP standard như C1/C3.

AM này **không** matched bake-off với Kaldi M13 hay HuBERT-base (kiến trúc / pretrain khác). Thesis ghi ΔPCC trên phone đã ghép cặp.


### Ghi chú resume / OOM / disk

- OOM: `BATCH_SIZE = 1`, `GRAD_ACCUM = 16`, hoặc hạ `MAX_SECONDS` xuống 12.
- Hết giờ session: `RESUME = True`, chạy lại từ cell Config (giữ `/kaggle/working/wav2vec2_phoneme_ctc/checkpoint-*`). Output Kaggle **không** giữ qua session mới trừ khi Save Version; download zip checkpoint nếu cần.
- Full 10 epoch: set `SMOKE = False` rồi Run All. Không train trên Speechocean762.
- Không dùng `jonatasgrosman/wav2vec2-large-xlsr-53-english` (CTC chữ cái, vocab khác 39 phone).
- Nếu Save Version chết với `high /tmp usage` / Docker daemon: system `/tmp` đầy (HF cache + `.map()` audio float32). Notebook này chuyển cache sang `/kaggle/tmp` và không materialize Arrow audio. **Restart session** rồi Run All (đừng resume kernel đã đầy `/tmp`).
- Checkpoint XLSR-53 + optimizer ~3–4GB/cái; `save_total_limit=2` để vừa 20GB `/kaggle/working`.
